# World Happiness Analysis & Prediction
### Python Data Analysis & Introductory Machine Learning Project

**Fix notes (this revision):** the dataset is now loaded and cleaned **once** at the
top of the notebook, and every later cell reuses that single cleaned `df`. In the
previous version, almost every cell independently re-read `dataset.csv` from disk —
meaning the cleaning done in the early cells never actually reached the charts or the
model, which were all built on the raw, unclean data. This version also adds Mean
Absolute Error (MAE) to the model evaluation, since the README already claimed it as
an evaluation metric but the original notebook never computed it.


In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Load the dataset ONCE — every later cell reuses this df
df = pd.read_csv("dataset.csv")

print("Shape:", df.shape)
df.dtypes


Shape: (157, 11)


Country                              str
Region                               str
Happiness Rank                     int64
Happiness Score                  float64
Economy (GDP per Capita)         float64
Family                           float64
Health (Life Expectancy)         float64
Freedom                          float64
Trust (Government Corruption)    float64
Generosity                       float64
Dystopia Residual                float64
dtype: object

## 1. Data Cleaning

Steps performed, and why:

- **Whitespace / empty strings:** country and region names can have stray whitespace
  from manual data entry; empty strings are converted to real `NaN` so pandas treats
  them as missing rather than as a valid category.
- **Dtype correction:** numeric columns are sometimes read as `object` if a single row
  has a stray non-numeric character — `pd.to_numeric(..., errors="coerce")` forces
  them to numeric and turns anything unparseable into `NaN` we can then handle.
- **Missing value imputation:** for the small number of missing numeric values, we
  fill with the column mean. This dataset (World Happiness Report) has no missing
  values in this vintage, but the code is defensive so it still works correctly if a
  different year's file (which does have gaps) is swapped in.


In [2]:
# Whitespace / empty-string cleanup
df['Country'] = df['Country'].astype("string").str.strip()
df['Region'] = df['Region'].replace(r'^\s*$', np.nan, regex=True)

# Force numeric columns to numeric dtype (coerce bad values to NaN instead of crashing)
numeric_cols = [
    "Happiness Score", "Economy (GDP per Capita)", "Family",
    "Health (Life Expectancy)", "Freedom", "Trust (Government Corruption)",
    "Generosity",
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Categorical columns
df['Country'] = df['Country'].astype('category')
df['Region'] = df['Region'].astype('category')

print("Missing values after cleaning:")
print(df[numeric_cols].isna().sum())


Missing values after cleaning:
Happiness Score                  0
Economy (GDP per Capita)         0
Family                           0
Health (Life Expectancy)         0
Freedom                          0
Trust (Government Corruption)    0
Generosity                       0
dtype: int64


In [3]:
# Mean-impute any remaining missing numeric values (defensive — see note above)
missing_before = df[numeric_cols].isna().sum().sum()
df[numeric_cols] = df[numeric_cols].apply(lambda col: col.fillna(col.mean()))
print(f"Imputed {missing_before} missing numeric values with column means")


Imputed 0 missing numeric values with column means


## 2. Exploratory Data Analysis

All charts below use the single cleaned `df` from Section 1 — no re-reading the raw
CSV, so cleaning actually reaches the visuals this time.


### 2.1 Top 10 Happiest Countries — GDP & Life Expectancy

In [4]:
top10 = (
    df.sort_values(by="Happiness Score", ascending=False)
      .head(10)
      [["Country", "Economy (GDP per Capita)", "Health (Life Expectancy)"]]
)

fig1 = go.Figure()
fig1.add_bar(x=top10["Country"], y=top10["Economy (GDP per Capita)"], name="GDP per Capita")
fig1.add_bar(x=top10["Country"], y=top10["Health (Life Expectancy)"], name="Healthy Life Expectancy")
fig1.update_layout(
    title="GDP per Capita and Healthy Life Expectancy of Top 10 Countries",
    xaxis_title="Country", yaxis_title="Value", barmode="group",
)
fig1.show()


### 2.2 Correlation Between Happiness Factors

In [5]:
corr_matrix = df[
    ["Economy (GDP per Capita)", "Family", "Health (Life Expectancy)",
     "Freedom", "Trust (Government Corruption)", "Generosity", "Happiness Score"]
].corr(numeric_only=True)

fig2 = px.imshow(
    corr_matrix, text_auto=True, color_continuous_scale="RdBu",
    title="Correlation Heatmap of Happiness Factors", width=800, height=600,
)
fig2.show()


### 2.3 Happiness vs GDP per Capita, by Region

In [6]:
fig3 = px.scatter(
    df, x="Economy (GDP per Capita)", y="Happiness Score", color="Region",
    title="Happiness Score vs GDP per Capita by Region",
    labels={"Economy (GDP per Capita)": "GDP per Capita", "Happiness Score": "Happiness Score"},
)
fig3.show()


### 2.4 Average Happiness Score by Region

In [7]:
region_happiness = df.groupby("Region", as_index=False, observed=True)["Happiness Score"].mean()

fig4 = px.pie(
    region_happiness, names="Region", values="Happiness Score",
    title="Average Happiness Score by Region",
)
fig4.show()


### 2.5 GDP per Capita — World Map

In [8]:
fig5 = px.choropleth(
    df, locations="Country", locationmode="country names",
    color="Economy (GDP per Capita)", hover_name="Country",
    hover_data={"Health (Life Expectancy)": True, "Economy (GDP per Capita)": True},
    color_continuous_scale="Viridis",
    title="GDP per Capita of Countries with Healthy Life Expectancy",
)
fig5.show()


/tmp/ipykernel_609/1512377446.py:1: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig5 = px.choropleth(


## 3. Export Interactive Dashboard

All four figures are combined into a single standalone HTML dashboard.


In [9]:
with open("dashboard.html", "w") as f:
    f.write(fig1.to_html(full_html=False, include_plotlyjs="cdn"))
    f.write(fig3.to_html(full_html=False, include_plotlyjs=False))
    f.write(fig4.to_html(full_html=False, include_plotlyjs=False))
    f.write(fig5.to_html(full_html=False, include_plotlyjs=False))

print("dashboard.html has been created successfully.")


dashboard.html has been created successfully.


## 4. Predictive Model — Linear Regression

Goal: estimate Happiness Score from the six underlying factors, and see which factor
carries the most weight. We reuse the already-cleaned `df` and its already-numeric
feature columns — no need to re-convert or re-impute.


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

features = [
    "Economy (GDP per Capita)", "Health (Life Expectancy)", "Family",
    "Freedom", "Trust (Government Corruption)", "Generosity",
]

X = df[features]
y = df["Happiness Score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("R² Score:", round(r2_score(y_test, y_pred), 4))
print("MAE:", round(mean_absolute_error(y_test, y_pred), 4))


R² Score: 0.707
MAE: 0.4992


In [11]:
importance = pd.Series(model.coef_, index=features).sort_values(ascending=False)
importance


Health (Life Expectancy)         1.513846
Freedom                          1.135663
Family                           1.071204
Economy (GDP per Capita)         0.844633
Trust (Government Corruption)    0.718714
Generosity                       0.395697
dtype: float64

## 5. Key Takeaways

- Health (Life Expectancy) and Freedom carry the largest positive coefficients in the
  model — moving those factors moves the predicted Happiness Score more than an equal
  change in GDP per capita does.
- GDP per capita is still positively associated with happiness, but the model
  confirms it is **not the dominant factor**, consistent with the project's original
  hypothesis.
- R² and MAE together give a fuller picture than R² alone: R² says how much
  variance the model explains, MAE says how far off a typical prediction is in the
  original 0-10 happiness-score units — useful when explaining model performance to a
  non-technical audience.

## 6. Limitations

- Small dataset (~150 countries) — a linear model with 6 features has limited room to
  overfit, but also limited data to learn subtle non-linear relationships.
- Happiness Score itself is a survey-based composite measure, not an objective ground
  truth, so "predicting" it is really predicting a specific survey methodology's
  output.
- Cross-country comparisons don't account for how culturally different populations
  interpret the same survey question.
